In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit import DataStructs
from tqdm import tqdm
import matplotlib.pyplot as plt


In [ ]:
sam_lit = pd.read_csv('./sam.csv')
sam_lit = sam_lit.rename(columns=str.lower)
trainset = pd.read_csv("../SDF_SMILES_Prop/Dataset_train_homo.csv")
trainset = trainset.rename(columns=str.lower)

In [ ]:
sam_lit.head()

In [ ]:
trainset.head()

In [ ]:
sam_lit_smiles = sam_lit['smiles']
trainset_smiles = trainset['smiles']

In [ ]:
class utils():
    def __init__(self):
        super().__init__()
        self.mol_from_smiles=Chem.MolFromSmiles
        self.fingerprint=Chem.AllChem.GetMorganFingerprintAsBitVect
        self.similarity = DataStructs.FingerprintSimilarity
    
    def mol(self,x):
        mol=self.mol_from_smiles(x)
        return mol

    def ECFP(self,mol):
        ecfp=self.fingerprint(mol, radius=2, nBits=2048)
        return ecfp    

In [ ]:
sam_lit_mol=[utils().mol(i) for i in sam_lit_smiles]
trainset_mol=[utils().mol(i) for i in trainset_smiles]
sam_lit_ecfp=[utils().ECFP(i) for  i in sam_lit_mol]
trainset_ecfp=[utils().ECFP(i) for i in trainset_mol]

In [ ]:
len(sam_lit_ecfp)

In [ ]:
len(trainset_ecfp)

In [ ]:
# calculate the similarity matrices of gen set and lit set
trainset_similarities=np.zeros((len(trainset_smiles),len(sam_lit_smiles)))
for i in tqdm(range(len(trainset_ecfp)), desc="Calculating similarities", unit="batch"):
    for j in range(len(sam_lit_ecfp)):
        score=DataStructs.FingerprintSimilarity(trainset_ecfp[i],sam_lit_ecfp[j])
        trainset_similarities[i,j]=score

In [ ]:
trainset_similarities.shape

In [ ]:
trainset_similarities

In [ ]:
trainset_average_similarity=np.sum(trainset_similarities,axis=1)/len(sam_lit_smiles)

In [ ]:
trainset_average_similarity.shape

In [ ]:
trainset_average_similarity

In [ ]:
trainset_max_similarity = np.max(trainset_similarities, axis=1)

In [ ]:
trainset_max_similarity.shape

In [ ]:
trainset_max_similarity

In [ ]:
df_ave = pd.DataFrame(trainset_average_similarity, columns=['Average_Similarity'])
df_ave.to_csv("./trainset_average_similarity.csv")

In [ ]:
# Scatter Plot
plt.figure(figsize=(10, 6))
plt.scatter(range(len(trainset_average_similarity)), trainset_average_similarity, label='trainset', alpha=0.5)
plt.legend()
plt.ylabel('trainset average similarity score')
plt.xlabel('Index')
plt.show()

In [ ]:
# calculate the similarity matrices of gen set and lit set
sam_similarities=np.zeros((len(sam_lit_smiles),len(sam_lit_smiles)))
for i in tqdm(range(len(sam_lit_ecfp)), desc="Calculating similarities", unit="batch"):
    for j in range(len(sam_lit_ecfp)):
        score=DataStructs.FingerprintSimilarity(sam_lit_ecfp[i],sam_lit_ecfp[j])
        sam_similarities[i,j]=score

In [ ]:
sam_similarities.shape

In [ ]:
sam_similarities

In [ ]:
sam_average_similarity=np.sum(sam_similarities,axis=1)/len(sam_lit_smiles)

In [ ]:
sam_average_similarity.shape

In [ ]:
df_sam_ave = pd.DataFrame(sam_average_similarity, columns=['Average_Similarity'])
df_sam_ave.to_csv("./sam_average_similarity.csv")

In [ ]:
# Scatter Plot
plt.figure(figsize=(10, 6))
plt.scatter(range(len(sam_average_similarity)), sam_average_similarity, label='sam', alpha=0.5)
plt.legend()
plt.ylabel('sam average similarity score')
plt.xlabel('Index')
plt.show()